# Quantum Many-Body Physics Exercises

This notebook provides interactive demonstrations of solutions to quantum many-body physics exercises, focusing on:

1. **Exercise 1**: Commutator relations for tight-binding Hamiltonians
2. **Exercise 2**: Tensor network constructions
3. **Exercise 3**: Gauge-invariant Hilbert space dimensions
4. **Exercise 4**: 2D Quantum spin ice models

---

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from quantum_many_body_exercises import (
    TightBindingModel,
    TensorNetworkConstructor,
    GaugeInvariantHilbertSpace,
    QuantumSpinIce2D
)

# Configure matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("✓ All modules imported successfully")

---
## Exercise 1: Commutator Relations for Tight-Binding Hamiltonian

### Problem Statement

For the Hamiltonian:

$$H = -t \sum_i (c_i^\dagger c_{i+1} + c_{i+1}^\dagger c_i)$$

We need to:
1. Check that $[H, \hat{N}] = 0$ (particle number conservation)
2. Find other operators that commute or don't commute with $\hat{N}$
3. Check that $[G, H] = 0$ for gauge operators

### Physical Interpretation

- The Hamiltonian describes **spinless fermions** hopping on a 1D lattice
- $c_i^\dagger$ and $c_i$ are fermionic creation/annihilation operators
- $\hat{N} = \sum_i c_i^\dagger c_i$ is the total particle number operator
- If $[H, \hat{N}] = 0$, particle number is **conserved**

In [ ]:
# Create tight-binding model
L = 4  # Number of sites
model = TightBindingModel(L=L, t=1.0, periodic=True)

# Get operators
H = model.hamiltonian()
N = model.total_number_operator()

print(f"System: {L} sites, spinless fermions")
print(f"Hilbert space dimension: {model.dim}")
print(f"\nHamiltonian shape: {H.shape}")
print(f"Number operator shape: {N.shape}")

### Part (a): Check $[H, \hat{N}] = 0$

In [ ]:
# Check commutator [H, N]
commutator_HN = model.commutator(H, N)
norm_HN = np.linalg.norm(commutator_HN)

print(f"[H, N] norm: {norm_HN:.2e}")
print(f"Commutes: {norm_HN < 1e-10}")

if norm_HN < 1e-10:
    print("\n✓ VERIFIED: The Hamiltonian commutes with total particle number.")
    print("  Physical interpretation: Particle number is conserved.")

### Part (b): Find Other Operators

In [ ]:
# Test various operators
print("Testing commutation with number operator N:\n")

# Local number operator
n_0 = model.number_operator_site(0)
model.check_commutation(N, n_0, "N", "n_0")

# Annihilation operator (should NOT commute)
c_0 = model.annihilation_operator(0)
model.check_commutation(N, c_0, "N", "c_0")

# Creation operator (should NOT commute)
c_dag_0 = model.creation_operator(0)
model.check_commutation(N, c_dag_0, "N", "c^dagger_0")

# H^2 (should commute)
H2 = H @ H
model.check_commutation(N, H2, "N", "H^2")

### Visualize the Hamiltonian and Number Operator

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot Hamiltonian
im1 = axes[0].imshow(np.real(H), cmap='RdBu', aspect='auto')
axes[0].set_title('Hamiltonian H (real part)', fontsize=14)
axes[0].set_xlabel('Basis state')
axes[0].set_ylabel('Basis state')
plt.colorbar(im1, ax=axes[0])

# Plot Number operator
im2 = axes[1].imshow(np.real(N), cmap='RdBu', aspect='auto')
axes[1].set_title('Number Operator N', fontsize=14)
axes[1].set_xlabel('Basis state')
axes[1].set_ylabel('Basis state')
plt.colorbar(im2, ax=axes[1])

# Plot commutator
im3 = axes[2].imshow(np.abs(commutator_HN), cmap='viridis', aspect='auto')
axes[2].set_title(f'|[H, N]| (norm={norm_HN:.2e})', fontsize=14)
axes[2].set_xlabel('Basis state')
axes[2].set_ylabel('Basis state')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

---
## Exercise 2: Tensor Network Construction

### Problem Statement

Write explicitly the tensor in equation (6.10):

$$S_{\alpha_2, \beta_1, \beta_2} = T_{\tau_2, \upsilon_1, \upsilon_2} \delta_{\ell_2 + m_1 + m_2, 0}$$

And extend the construction to:
1. A third site
2. Generalize numerically to L sites using **Matrix Product States (MPS)**

### Physical Interpretation

- Tensor networks provide efficient representations of many-body quantum states
- The Kronecker delta enforces a **gauge constraint**
- MPS offers exponential compression for 1D systems

In [ ]:
# Create tensor network constructor
constructor = TensorNetworkConstructor(spin=1)

print(f"Spin: {constructor.spin}")
print(f"Local dimension: {constructor.local_dim}")

### Part (a): Explicit Two-Site Tensor

In [ ]:
# Create two-site tensor with gauge constraint
T2 = constructor.create_simple_tensor(constraint_value=0)

print(f"Two-site tensor shape: {T2.shape}")
print(f"Number of non-zero elements: {np.count_nonzero(T2)}")
print(f"Tensor norm: {np.linalg.norm(T2):.4f}")

# Visualize tensor slices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, ax in enumerate(axes):
    im = ax.imshow(np.abs(T2[i, :, :]), cmap='viridis')
    ax.set_title(f'|T[{i}, :, :]|', fontsize=14)
    ax.set_xlabel('Index 2')
    ax.set_ylabel('Index 1')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

### Part (b): Extension to Three Sites

In [ ]:
# Create three-site tensor
T3 = constructor.extend_to_three_sites()

print(f"Three-site tensor shape: {T3.shape}")
print(f"Number of non-zero elements: {np.count_nonzero(T3)}")
print(f"Tensor norm: {np.linalg.norm(T3):.4f}")
print(f"Sparsity: {1 - np.count_nonzero(T3) / T3.size:.4f}")

### Part (c): MPS Representation for L Sites

In [ ]:
# Compare MPS vs exact representation for different system sizes
L_values = [4, 6, 8, 10, 12, 14]
bond_dim = 4

results = []
for L in L_values:
    mps = constructor.construct_mps_chain(L, bond_dim)
    
    total_params = sum(tensor.size for tensor in mps)
    exact_params = constructor.local_dim ** L
    compression = exact_params / total_params
    
    results.append({
        'L': L,
        'mps_params': total_params,
        'exact_params': exact_params,
        'compression': compression
    })

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

L_plot = [r['L'] for r in results]
mps_plot = [r['mps_params'] for r in results]
exact_plot = [r['exact_params'] for r in results]
compression_plot = [r['compression'] for r in results]

ax1.semilogy(L_plot, mps_plot, 'o-', label='MPS', linewidth=2, markersize=8)
ax1.semilogy(L_plot, exact_plot, 's-', label='Exact', linewidth=2, markersize=8)
ax1.set_xlabel('Number of sites (L)', fontsize=12)
ax1.set_ylabel('Number of parameters', fontsize=12)
ax1.set_title('MPS vs Exact Representation', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.semilogy(L_plot, compression_plot, 'o-', color='green', linewidth=2, markersize=8)
ax2.set_xlabel('Number of sites (L)', fontsize=12)
ax2.set_ylabel('Compression ratio', fontsize=12)
ax2.set_title('Exponential Compression by MPS', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nCompression ratios:")
for r in results:
    print(f"  L={r['L']:2d}: {r['compression']:10.2e}")

---
## Exercise 3: Gauge Invariant Hilbert Space Dimension

### Problem Statement

Using relation (6.26):

$$KP_{N'}|\psi\rangle = KPP_{N'}|\psi\rangle = KP_{N'}P|\psi\rangle = KP_{N'}K^\dagger K|\psi\rangle = P_{N'}^G|\psi^G\rangle$$

Evaluate numerically the dimension of the gauge invariant Hilbert space for 1D spin-1 sector as a function of the number of sites.

### Physical Interpretation

- Gauge constraints reduce the Hilbert space dimension
- This is crucial for efficient numerical simulations
- Understanding scaling helps predict computational requirements

In [ ]:
# Create gauge invariant Hilbert space calculator
calculator = GaugeInvariantHilbertSpace(spin=1)

print(f"Spin: {calculator.spin}")
print(f"Local dimension: {calculator.local_dim}")
print("Gauge constraint: Total quantum number = 0\n")

### Compute Dimensions vs L

In [ ]:
# Compute dimensions for different L values
results = calculator.compute_dimensions_vs_L(L_max=8, constraint_type='total_zero')

### Visualize Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Dimensions vs L
ax = axes[0, 0]
ax.semilogy(results['L'], results['dim_gauge'], 'o-', label='Gauge invariant',
           linewidth=2, markersize=8)
ax.semilogy(results['L'], results['dim_total'], 's-', label='Total',
           linewidth=2, markersize=8)
ax.set_xlabel('Number of sites (L)', fontsize=12)
ax.set_ylabel('Hilbert space dimension', fontsize=12)
ax.set_title('Hilbert Space Dimensions', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Plot 2: Ratio vs L
ax = axes[0, 1]
ax.plot(results['L'], results['ratio'], 'o-', color='green',
       linewidth=2, markersize=8)
ax.set_xlabel('Number of sites (L)', fontsize=12)
ax.set_ylabel('Ratio (gauge / total)', fontsize=12)
ax.set_title('Gauge Constraint Ratio', fontsize=14)
ax.grid(True, alpha=0.3)

# Plot 3: Log-log plot
ax = axes[1, 0]
ax.loglog(results['L'], results['dim_gauge'], 'o-', label='Gauge invariant',
         linewidth=2, markersize=8)
ax.loglog(results['L'], results['dim_total'], 's-', label='Total',
         linewidth=2, markersize=8)
ax.set_xlabel('Number of sites (L)', fontsize=12)
ax.set_ylabel('Hilbert space dimension', fontsize=12)
ax.set_title('Scaling Analysis (log-log)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Plot 4: Reduction factor
ax = axes[1, 1]
reduction_factor = results['dim_total'] / results['dim_gauge']
ax.semilogy(results['L'], reduction_factor, 'o-', color='red',
           linewidth=2, markersize=8)
ax.set_xlabel('Number of sites (L)', fontsize=12)
ax.set_ylabel('Reduction factor (total / gauge)', fontsize=12)
ax.set_title('Computational Advantage', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Analysis

In [ ]:
print("Analysis of gauge invariant Hilbert space:")
print("=" * 50)

avg_ratio = np.mean(results['ratio'])
print(f"\nAverage ratio (gauge/total): {avg_ratio:.4f}")
print(f"Final ratio at L={results['L'][-1]}: {results['ratio'][-1]:.4f}")

# Growth rate
if len(results['L']) >= 3:
    growth_gauge = np.mean(np.diff(np.log(results['dim_gauge'])) / np.diff(results['L']))
    growth_total = np.mean(np.diff(np.log(results['dim_total'])) / np.diff(results['L']))
    
    print(f"\nAverage growth rate (gauge): {growth_gauge:.4f}")
    print(f"Average growth rate (total): {growth_total:.4f}")
    print(f"Expected total growth: {np.log(calculator.local_dim):.4f}")

print("\nConclusion:")
print("- Gauge constraints significantly reduce Hilbert space dimension")
print("- Computational advantage grows with system size")
print("- Critical for large-scale quantum simulations")

---
## Exercise 4: 2D Quantum Spin Ice Model

### Problem Statement

Compute the operators **K** and **P_{N'}** for the two-dimensional quantum spin ice model.

### Physical Interpretation

- Quantum spin ice models frustrated magnetism
- Spins live on **links** (edges) of the lattice
- Ice rule: $\sum_{\text{links around plaquette}} S_z = 0$
- Emergent gauge theory with topological order

In [ ]:
# Create 2D quantum spin ice model
Lx, Ly = 4, 4
model = QuantumSpinIce2D(Lx, Ly)

print(f"System: {Lx} × {Ly} lattice")
print(f"Number of sites: {model.N_sites}")
print(f"Number of links (spins): {model.N_links}")
print(f"Number of plaquettes: {(Lx-1) * (Ly-1)}")

### Visualize the Lattice

In [ ]:
fig, ax = model.visualize_lattice()
plt.show()

### Gauge Operators and Projectors

In [ ]:
print("Gauge Constraint Operator G:")
print("=" * 50)
model.construct_gauge_constraint_operator()

print("\n" + "=" * 50)
print("\nParticle Number Projector P_{N'}:")
print("=" * 50)
N_target = model.N_links // 2
model.construct_particle_number_projector(N_target)

print("\n" + "=" * 50)
print("\nGauge Projector K:")
print("=" * 50)
model.construct_gauge_projector_K()

### Gauge-Invariant Dimension

In [ ]:
dim_gauge = model.compute_gauge_invariant_dimension()

### Scaling Analysis

In [ ]:
# Analyze scaling with system size
sizes = [(2, 2), (2, 3), (3, 3), (3, 4), (4, 4), (4, 5), (5, 5)]

scaling_data = []
for Lx, Ly in sizes:
    model_temp = QuantumSpinIce2D(Lx, Ly)
    N_plaq = (Lx - 1) * (Ly - 1)
    dim_total = 2 ** model_temp.N_links
    dim_gauge_est = 2 ** (model_temp.N_links - N_plaq + 1)
    
    scaling_data.append({
        'Lx': Lx,
        'Ly': Ly,
        'N_links': model_temp.N_links,
        'N_plaq': N_plaq,
        'dim_total': dim_total,
        'dim_gauge': dim_gauge_est,
        'ratio': dim_gauge_est / dim_total
    })

# Plot scaling
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# System size vs dimensions
system_sizes = [d['N_links'] for d in scaling_data]
dim_totals = [d['dim_total'] for d in scaling_data]
dim_gauges = [d['dim_gauge'] for d in scaling_data]

ax1.semilogy(system_sizes, dim_totals, 's-', label='Total', linewidth=2, markersize=8)
ax1.semilogy(system_sizes, dim_gauges, 'o-', label='Gauge invariant',
            linewidth=2, markersize=8)
ax1.set_xlabel('Number of links', fontsize=12)
ax1.set_ylabel('Hilbert space dimension', fontsize=12)
ax1.set_title('2D Quantum Spin Ice Scaling', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Ratio vs system size
ratios = [d['ratio'] for d in scaling_data]
ax2.semilogy(system_sizes, ratios, 'o-', color='green', linewidth=2, markersize=8)
ax2.set_xlabel('Number of links', fontsize=12)
ax2.set_ylabel('Ratio (gauge / total)', fontsize=12)
ax2.set_title('Gauge Constraint Ratio', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print table
print("\nScaling table:")
print(f"{'Lx':>4} {'Ly':>4} {'N_links':>8} {'N_plaq':>8} {'dim_gauge':>12} {'ratio':>10}")
print("=" * 60)
for d in scaling_data:
    print(f"{d['Lx']:4d} {d['Ly']:4d} {d['N_links']:8d} {d['N_plaq']:8d} "
          f"{d['dim_gauge']:12d} {d['ratio']:10.6f}")

---
## Summary and Conclusions

### Exercise 1: Commutator Relations
- ✓ Verified $[H, \hat{N}] = 0$: particle number is conserved
- ✓ Local observables commute with $\hat{N}$
- ✓ Creation/annihilation operators do NOT commute with $\hat{N}$

### Exercise 2: Tensor Networks
- ✓ Constructed explicit tensors with gauge constraints
- ✓ Extended to three sites
- ✓ MPS provides exponential compression for large L

### Exercise 3: Gauge Invariant Hilbert Space
- ✓ Gauge constraints significantly reduce Hilbert space dimension
- ✓ Computational advantage grows exponentially with system size
- ✓ Critical for large-scale quantum simulations

### Exercise 4: 2D Quantum Spin Ice
- ✓ Understood gauge constraint operators G
- ✓ Analyzed particle number projectors P_{N'}
- ✓ Computed gauge projectors K
- ✓ Demonstrated emergent gauge theory structure

### Key Takeaways

1. **Symmetries and Conservation Laws**: Commutator relations reveal conservation laws
2. **Tensor Networks**: Efficient representations of many-body quantum states
3. **Gauge Theories**: Constraints reduce Hilbert space and enable simulations
4. **Emergent Physics**: Complex phenomena emerge from simple local rules

---